In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_7.pickle

In [4]:
%%RecordEvent
%%time
### cell 8 ###

### cell 8 optimized
cols = ['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']

# Mobile stats: do first and last in one groupby.agg
mobile_agg = Mobile_df.groupby('Name')[cols].agg(['first','last'])
# split out first/last frames
first_m = mobile_agg.xs('first', level=1, axis=1)
last_m  = mobile_agg.xs('last',  level=1, axis=1)
Mobile_Stats = (
    (last_m['Avg. Avg D Kbps'] - first_m['Avg. Avg D Kbps'])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(last_m['Avg. Avg U Kbps'] - first_m['Avg. Avg U Kbps']),
        Change_Latency=(last_m['Avg Lat Ms']      - first_m['Avg Lat Ms'])
    )
)

# Broadband stats: same pattern
broad_agg = Broadband_df.groupby('Name')[cols].agg(['first','last'])
first_b = broad_agg.xs('first', level=1, axis=1)
last_b  = broad_agg.xs('last',  level=1, axis=1)
Broadband_Stats = (
    (last_b['Avg. Avg D Kbps'] - first_b['Avg. Avg D Kbps'])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(last_b['Avg. Avg U Kbps'] - first_b['Avg. Avg U Kbps']),
        Change_Latency=(last_b['Avg Lat Ms']      - first_b['Avg Lat Ms'])
    )
)

# combine only the download‐change columns
Total_Stats = (
    Mobile_Stats['Change_Download'].to_frame('Mobile')
    .join(Broadband_Stats['Change_Download'].to_frame('Broadband'))
)

CPU times: user 101 ms, sys: 50.1 ms, total: 151 ms
Wall time: 139 ms


In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_8_try_1.pickle